# Pré-processamento: Correlação de Pearson + PCA
Bases: **Wine**, **Semeion** e **Image**.

Etapas:
1. Correlação de Pearson entre atributos (top 20 pares mais correlatos, limiar 0,80)
2. Remoção de 1 atributo por par com correlação >= 0,80 (dentro do top 20)
3. PCA (nº de componentes = raiz(n) e >= 90% de variância explicada), aplicado na base original e na base pós-correlação
4. Salvamento das 9 bases resultantes (3 correlação + 6 PCA)

In [7]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import os

DATA_DIR = "dataset"
OUT_DIR = "outputs"
os.makedirs(OUT_DIR, exist_ok=True)

# base: (arquivo, coluna alvo/classe)
DATASETS = {
    "Wine": ("Wine.csv", "class"),
    "Semeion": ("Semeion.csv", "class"),
    "Image": ("Image.csv", "Class"),
}

## 1. Carregamento das bases

In [9]:
dfs = {}
for nome, (arquivo, target) in DATASETS.items():
    dfs[nome] = pd.read_csv(os.path.join(DATA_DIR, arquivo))
    print(nome, dfs[nome].shape)

Wine (4898, 12)
Semeion (1593, 257)
Image (2310, 20)


## 2. Correlação de Pearson - top 20 pares mais correlatos

In [10]:
def top20_correlacionados(df, target_col):
    # correlação de Pearson (em módulo) apenas entre os atributos, exclui a classe
    X = df.drop(columns=[target_col])
    corr = X.corr(method="pearson").abs()
    cols = corr.columns
    pares = []
    for i in range(len(cols)):
        for j in range(i + 1, len(cols)):
            v = corr.iloc[i, j]
            if not np.isnan(v):  # ignora atributos constantes (corr indefinida)
                pares.append((cols[i], cols[j], v))
    # ordena por correlação decrescente e mantém os 20 maiores
    pares_top20 = sorted(pares, key=lambda x: x[2], reverse=True)[:20]
    return pd.DataFrame(pares_top20, columns=["atributo_1", "atributo_2", "correlacao"])

In [11]:
top20 = {}
for nome, (_, target) in DATASETS.items():
    top20[nome] = top20_correlacionados(dfs[nome], target)
    print(f"\nTop 20 pares mais correlatos - {nome}")
    display(top20[nome])


Top 20 pares mais correlatos - Wine


,atributo_1,atributo_2,correlacao
0,residualsugar,density,0.838966
1,density,alcohol,0.780138
2,freesulfur,totalsulfur,0.615501
3,totalsulfur,density,0.529881
4,residualsugar,alcohol,0.450631
5,totalsulfur,alcohol,0.448892
6,fixedacid,pH,0.425858
7,residualsugar,totalsulfur,0.401439
8,chlorides,alcohol,0.360189
9,residualsugar,freesulfur,0.299098



Top 20 pares mais correlatos - Semeion


,atributo_1,atributo_2,correlacao
0,a_6,a_7,0.808771
1,a_190,a_206,0.798533
2,a_5,a_6,0.798287
3,a_46,a_62,0.797173
4,a_174,a_190,0.794071
5,a_176,a_177,0.790442
6,a_4,a_5,0.789421
7,a_144,a_145,0.786870
8,a_2,a_3,0.782672
9,a_160,a_161,0.780703



Top 20 pares mais correlatos - Image


,atributo_1,atributo_2,correlacao
0,att12,att17,0.998644
1,att10,att11,0.998112
2,att10,att17,0.997385
3,att10,att13,0.995842
4,att10,att12,0.995809
5,att11,att13,0.994056
6,att11,att17,0.992062
7,att11,att12,0.990813
8,att13,att17,0.990042
9,att12,att13,0.984659


## 3. Remoção de atributos correlacionados (limiar 0,80)

In [12]:
def remover_correlacionados(df, target_col, top20_df, limiar=0.80):
    # dentro do top 20, remove apenas 1 atributo de cada par com correlação >= limiar
    to_drop = set()
    for _, row in top20_df.iterrows():
        if row["correlacao"] >= limiar and row["atributo_2"] not in to_drop:
            to_drop.add(row["atributo_2"])
    df_reduzido = df.drop(columns=list(to_drop))
    return df_reduzido, to_drop

In [13]:
dfs_corr = {}
for nome, (_, target) in DATASETS.items():
    df_reduzido, removidos = remover_correlacionados(dfs[nome], target, top20[nome])
    dfs_corr[nome] = df_reduzido
    print(f"{nome}: removidos {sorted(removidos)} -> shape final {df_reduzido.shape}")

Wine: removidos ['density'] -> shape final (4898, 11)
Semeion: removidos ['a_7'] -> shape final (1593, 256)
Image: removidos ['att11', 'att12', 'att13', 'att14', 'att15', 'att16', 'att17', 'att19'] -> shape final (2310, 12)


## 4. Salvando as 3 bases pós-correlação

In [14]:
for nome, df_c in dfs_corr.items():
    df_c.to_csv(os.path.join(OUT_DIR, f"{nome}_correlacao.csv"), index=False)
print("Bases de correlação salvas.")

Bases de correlação salvas.


## 5. PCA
Número de componentes definido como o **maior valor** entre `round(sqrt(n))` (n = nº de instâncias) e o mínimo de componentes necessário para reter >= 90% da variância, de forma a satisfazer as duas exigências simultaneamente.

In [15]:
def aplicar_pca(df, target_col):
    X = df.drop(columns=[target_col]).values
    y = df[target_col].values
    n = X.shape[0]

    X_scaled = StandardScaler().fit_transform(X)

    k_sqrt = int(round(np.sqrt(n)))
    variancia_acumulada = np.cumsum(PCA().fit(X_scaled).explained_variance_ratio_)
    k_90 = int(np.argmax(variancia_acumulada >= 0.90)) + 1
    # nº de componentes não pode exceder o nº de atributos disponíveis
    n_componentes = min(max(k_sqrt, k_90), X.shape[1])

    pca = PCA(n_components=n_componentes)
    X_pca = pca.fit_transform(X_scaled)

    df_pca = pd.DataFrame(X_pca, columns=[f"PC{i+1}" for i in range(n_componentes)])
    df_pca[target_col] = y

    print(f"n={n} | sqrt(n)={k_sqrt} | componentes p/ 90%={k_90} | "
          f"componentes usados={n_componentes} | variância explicada="
          f"{pca.explained_variance_ratio_.sum():.4f}")
    return df_pca

### 5.1 PCA sobre as bases originais (3 bases)

In [16]:
dfs_pca_original = {}
for nome, (_, target) in DATASETS.items():
    print(nome)
    dfs_pca_original[nome] = aplicar_pca(dfs[nome], target)
    dfs_pca_original[nome].to_csv(os.path.join(OUT_DIR, f"{nome}_pca_original.csv"), index=False)

Wine
n=4898 | sqrt(n)=70 | componentes p/ 90%=8 | componentes usados=11 | variância explicada=1.0000
Semeion
n=1593 | sqrt(n)=40 | componentes p/ 90%=111 | componentes usados=111 | variância explicada=0.8991
Image
n=2310 | sqrt(n)=48 | componentes p/ 90%=8 | componentes usados=19 | variância explicada=1.0000


### 5.2 PCA sobre as bases pós-correlação (3 bases)

In [17]:
dfs_pca_corr = {}
for nome, (_, target) in DATASETS.items():
    print(nome)
    dfs_pca_corr[nome] = aplicar_pca(dfs_corr[nome], target)
    dfs_pca_corr[nome].to_csv(os.path.join(OUT_DIR, f"{nome}_pca_correlacao.csv"), index=False)

Wine
n=4898 | sqrt(n)=70 | componentes p/ 90%=8 | componentes usados=10 | variância explicada=1.0000
Semeion
n=1593 | sqrt(n)=40 | componentes p/ 90%=111 | componentes usados=111 | variância explicada=0.8993
Image
n=2310 | sqrt(n)=48 | componentes p/ 90%=7 | componentes usados=11 | variância explicada=1.0000


## 6. Resumo das 9 bases salvas

In [18]:
print("Bases de correlação:")
for nome in DATASETS:
    print(f" - {nome}_correlacao.csv -> {dfs_corr[nome].shape}")

print("\nBases de PCA (originais):")
for nome in DATASETS:
    print(f" - {nome}_pca_original.csv -> {dfs_pca_original[nome].shape}")

print("\nBases de PCA (pós-correlação):")
for nome in DATASETS:
    print(f" - {nome}_pca_correlacao.csv -> {dfs_pca_corr[nome].shape}")

Bases de correlação:
 - Wine_correlacao.csv -> (4898, 11)
 - Semeion_correlacao.csv -> (1593, 256)
 - Image_correlacao.csv -> (2310, 12)

Bases de PCA (originais):
 - Wine_pca_original.csv -> (4898, 12)
 - Semeion_pca_original.csv -> (1593, 112)
 - Image_pca_original.csv -> (2310, 20)

Bases de PCA (pós-correlação):
 - Wine_pca_correlacao.csv -> (4898, 11)
 - Semeion_pca_correlacao.csv -> (1593, 112)
 - Image_pca_correlacao.csv -> (2310, 12)
